# The Complete Transformer Encoder-Decoder

This notebook assembles the components developed in notebook 3 into a complete sequence-to-sequence Transformer. It adds:

- a reusable English–German parallel-text dataset,
- dynamic per-batch padding and teacher-forcing shifts,
- stacked encoder and decoder blocks,
- shared source/target token embeddings and a tied output projection, and
- padding-aware next-token cross-entropy loss.

The implementation follows the post-layer-normalization structure of the original Transformer.

In [1]:
from itertools import islice, zip_longest
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tokenizers import Tokenizer

## 1. Configuration and tokenizer

The original Transformer used six encoder and six decoder layers. We set `num_layers = 6` to match the paper's depth. But in order to make a forward and backward pass practical on a CPU, use two layers. 

In [2]:
embedding_dim = 512
num_heads = 8
num_layers = 6
context_length = 128
dropout_rate = 0.1
label_smoothing = 0.1
batch_size = 4
max_examples = 10_000


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("Could not find repo root containing pyproject.toml")


repo_root = find_repo_root()
tokenizer_path = repo_root / "data" / "tokenizers" / "tatoeba_bpe_en_de.json"
english_path = repo_root / "data" / "raw" / "tatoeba_en_de" / "Tatoeba.de-en.en"
german_path = repo_root / "data" / "raw" / "tatoeba_en_de" / "Tatoeba.de-en.de"

tokenizer = Tokenizer.from_file(str(tokenizer_path))
vocab_size = tokenizer.get_vocab_size()
pad_token_id = tokenizer.token_to_id("[PAD]")
bos_token_id = tokenizer.token_to_id("[BOS]")
eos_token_id = tokenizer.token_to_id("[EOS]")

assert None not in (pad_token_id, bos_token_id, eos_token_id)
print(f"Vocabulary size: {vocab_size:,}")
print(f"PAD/BOS/EOS ids: {pad_token_id}/{bos_token_id}/{eos_token_id}")

Vocabulary size: 16,000
PAD/BOS/EOS ids: 0/1/2


## 2. Parallel dataset and dynamic batching

The two Tatoeba files are line-aligned: English line `i` is paired with German line `i`. The dataset stores text pairs and tokenizes examples on demand. The collator then pads only to the longest source and target in the current batch.

Truncation retains `[EOS]`, which is important because it teaches the decoder when to stop generating.

In [3]:
class ParallelTextDataset(Dataset):
    def __init__(
        self,
        source_path,
        target_path,
        tokenizer,
        max_seq_len,
        max_examples=None,
    ):
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.eos_token_id = tokenizer.token_to_id("[EOS]")

        pairs = []
        with (
            open(source_path, encoding="utf-8") as source_file,
            open(target_path, encoding="utf-8") as target_file,
        ):
            lines = zip_longest(source_file, target_file)
            if max_examples is not None:
                lines = islice(lines, max_examples)

            for source_line, target_line in lines:
                if source_line is None or target_line is None:
                    raise ValueError(
                        "Source and target files must contain the same number of lines"
                    )
                source_text = source_line.rstrip()
                target_text = target_line.rstrip()
                if source_text and target_text:
                    pairs.append((source_text, target_text))

        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def _encode(self, text):
        token_ids = self.tokenizer.encode(text).ids
        if len(token_ids) > self.max_seq_len:
            token_ids = token_ids[: self.max_seq_len - 1] + [self.eos_token_id]
        return token_ids

    def __getitem__(self, index):
        source_text, target_text = self.pairs[index]
        return {
            "source_ids": self._encode(source_text),
            "target_ids": self._encode(target_text),
        }


class TranslationCollator:
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id

    def _pad(self, sequences):
        max_len = max(len(sequence) for sequence in sequences)
        padded = [
            sequence + [self.pad_token_id] * (max_len - len(sequence))
            for sequence in sequences
        ]
        return torch.tensor(padded, dtype=torch.long)

    def __call__(self, examples):
        source_ids = self._pad([example["source_ids"] for example in examples])
        target_ids = self._pad([example["target_ids"] for example in examples])

        decoder_input_ids = target_ids[:, :-1]
        labels = target_ids[:, 1:]

        return {
            "source_ids": source_ids,
            "decoder_input_ids": decoder_input_ids,
            "labels": labels,
            "source_padding_mask": source_ids.eq(self.pad_token_id),
            "decoder_padding_mask": decoder_input_ids.eq(self.pad_token_id),
        }

In [4]:
train_dataset = ParallelTextDataset(
    source_path=english_path,
    target_path=german_path,
    tokenizer=tokenizer,
    max_seq_len=context_length,
    max_examples=max_examples,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=TranslationCollator(pad_token_id),
)

batch = next(iter(train_loader))
for name, tensor in batch.items():
    print(f"{name:22s} {tuple(tensor.shape)}")

source_ids             (4, 15)
decoder_input_ids      (4, 13)
labels                 (4, 13)
source_padding_mask    (4, 15)
decoder_padding_mask   (4, 13)


## 3. Components from notebook 3

These are the reusable positional encoding, multi-head attention, encoder-block, and decoder-block definitions. The standalone component tests from notebook 3 are intentionally omitted.

In [5]:
class SinusoidalPositionEmbedding(nn.Module):
    def __init__(self, context_length, embedding_dim):
        super().__init__()
        position = torch.arange(context_length).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, embedding_dim, 2)
            * (-torch.log(torch.tensor(10_000.0)) / embedding_dim)
        )

        pe = torch.zeros(context_length, embedding_dim)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, positions):
        return self.pe[positions]


class MultiheadAttention(nn.Module):
    def __init__(
        self,
        embedding_dim,
        num_heads,
        context_length,
        is_crossattention=False,
        is_causal=False,
        dropout_rate=0.1,
    ):
        super().__init__()
        assert embedding_dim % num_heads == 0
        assert not (is_crossattention and is_causal)

        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.head_dim = embedding_dim // num_heads
        self.is_crossattention = is_crossattention
        self.is_causal = is_causal

        self.w_q = nn.Linear(embedding_dim, embedding_dim)
        self.w_k = nn.Linear(embedding_dim, embedding_dim)
        self.w_v = nn.Linear(embedding_dim, embedding_dim)
        self.w_o = nn.Linear(embedding_dim, embedding_dim)
        self.attn_dropout = nn.Dropout(dropout_rate)

        if is_causal:
            causal_mask = torch.triu(
                torch.ones(context_length, context_length, dtype=torch.bool),
                diagonal=1,
            )
            self.register_buffer("causal_mask", causal_mask)

    def forward(self, hidden_states, key_value_states=None, padding_mask=None):
        batch_size, query_len, embedding_dim = hidden_states.shape
        assert embedding_dim == self.embedding_dim

        if self.is_crossattention:
            assert key_value_states is not None
        else:
            assert key_value_states is None
            key_value_states = hidden_states

        assert key_value_states.shape[0] == batch_size
        assert key_value_states.shape[2] == self.embedding_dim
        key_value_len = key_value_states.shape[1]

        q = self.w_q(hidden_states)
        k = self.w_k(key_value_states)
        v = self.w_v(key_value_states)

        q = q.view(batch_size, query_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, key_value_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, key_value_len, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = q @ k.transpose(-2, -1) / (self.head_dim**0.5)

        if self.is_causal:
            assert query_len <= self.causal_mask.shape[0]
            assert key_value_len <= self.causal_mask.shape[1]
            causal_mask = self.causal_mask[:query_len, :key_value_len]
            attn_scores = attn_scores.masked_fill(
                causal_mask[None, None, :, :], float("-inf")
            )

        if padding_mask is not None:
            assert padding_mask.shape == (batch_size, key_value_len)
            assert padding_mask.dtype == torch.bool
            attn_scores = attn_scores.masked_fill(
                padding_mask[:, None, None, :], float("-inf")
            )

        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)
        output = attn_weights @ v
        output = output.transpose(1, 2).contiguous().view(batch_size, query_len, embedding_dim)
        return self.w_o(output)

In [6]:
class EncoderBlock(nn.Module):
    def __init__(self, embedding_dim, num_heads, context_length, dropout_rate=0.1):
        super().__init__()
        self.self_attention = MultiheadAttention(
            embedding_dim, num_heads, context_length, dropout_rate=dropout_rate
        )
        self.ffn = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim),
            nn.ReLU(),
            nn.Linear(4 * embedding_dim, embedding_dim),
        )
        self.attn_dropout = nn.Dropout(dropout_rate)
        self.ffn_dropout = nn.Dropout(dropout_rate)
        self.layer_norm1 = nn.LayerNorm(embedding_dim)
        self.layer_norm2 = nn.LayerNorm(embedding_dim)

    def forward(self, hidden_states, padding_mask=None):
        attn_output = self.self_attention(hidden_states, padding_mask=padding_mask)
        hidden_states = self.layer_norm1(
            hidden_states + self.attn_dropout(attn_output)
        )
        ffn_output = self.ffn(hidden_states)
        return self.layer_norm2(hidden_states + self.ffn_dropout(ffn_output))


class DecoderBlock(nn.Module):
    def __init__(self, embedding_dim, num_heads, context_length, dropout_rate=0.1):
        super().__init__()
        self.causal_self_attention = MultiheadAttention(
            embedding_dim,
            num_heads,
            context_length,
            is_causal=True,
            dropout_rate=dropout_rate,
        )
        self.cross_attention = MultiheadAttention(
            embedding_dim,
            num_heads,
            context_length,
            is_crossattention=True,
            dropout_rate=dropout_rate,
        )
        self.ffn = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim),
            nn.ReLU(),
            nn.Linear(4 * embedding_dim, embedding_dim),
        )
        self.self_attn_dropout = nn.Dropout(dropout_rate)
        self.cross_attn_dropout = nn.Dropout(dropout_rate)
        self.ffn_dropout = nn.Dropout(dropout_rate)
        self.layer_norm1 = nn.LayerNorm(embedding_dim)
        self.layer_norm2 = nn.LayerNorm(embedding_dim)
        self.layer_norm3 = nn.LayerNorm(embedding_dim)

    def forward(
        self,
        hidden_states,
        encoder_hidden_states,
        decoder_padding_mask=None,
        encoder_padding_mask=None,
    ):
        attn_output = self.causal_self_attention(
            hidden_states, padding_mask=decoder_padding_mask
        )
        hidden_states = self.layer_norm1(
            hidden_states + self.self_attn_dropout(attn_output)
        )

        attn_output = self.cross_attention(
            hidden_states,
            key_value_states=encoder_hidden_states,
            padding_mask=encoder_padding_mask,
        )
        hidden_states = self.layer_norm2(
            hidden_states + self.cross_attn_dropout(attn_output)
        )

        ffn_output = self.ffn(hidden_states)
        return self.layer_norm3(hidden_states + self.ffn_dropout(ffn_output))

## 4. Stacked encoder and decoder

Every layer preserves the sequence length and embedding dimension. The encoder converts source embeddings into contextual source representations. The decoder repeatedly refines its target-side states using causal self-attention and cross-attention to the final encoder states.

In [7]:
class TransformerEncoder(nn.Module):
    def __init__(
            self,
            embedding_dim,
            num_heads,
            context_length,
            num_layers,
            dropout_rate=0.1
    ):
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderBlock(
                embedding_dim=embedding_dim,
                num_heads=num_heads,
                context_length=context_length,
                dropout_rate=dropout_rate,
            )
            for _ in range(num_layers)
        ])

    def forward(self, hidden_states, padding_mask: torch.Tensor = None):
        for layer in self.layers:
            hidden_states = layer(hidden_states, padding_mask=padding_mask)

        return hidden_states
    
# testing
encoder = TransformerEncoder(512, 8, 128, 6, 0.1)
encoder

TransformerEncoder(
  (layers): ModuleList(
    (0-5): 6 x EncoderBlock(
      (self_attention): MultiheadAttention(
        (w_q): Linear(in_features=512, out_features=512, bias=True)
        (w_k): Linear(in_features=512, out_features=512, bias=True)
        (w_v): Linear(in_features=512, out_features=512, bias=True)
        (w_o): Linear(in_features=512, out_features=512, bias=True)
        (attn_dropout): Dropout(p=0.1, inplace=False)
      )
      (ffn): Sequential(
        (0): Linear(in_features=512, out_features=2048, bias=True)
        (1): ReLU()
        (2): Linear(in_features=2048, out_features=512, bias=True)
      )
      (attn_dropout): Dropout(p=0.1, inplace=False)
      (ffn_dropout): Dropout(p=0.1, inplace=False)
      (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
      (layer_norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
    )
  )
)

In [8]:
class TransformerDecoder(nn.Module):
    def __init__(
            self,
            embedding_dim,
            num_heads,
            context_length,
            num_layers,
            dropout_rate=0.1
    ):
        super().__init__()
        self.layers = nn.ModuleList([
            DecoderBlock(
                embedding_dim=embedding_dim,
                num_heads=num_heads,
                context_length=context_length,
                dropout_rate=dropout_rate,
            )
            for _ in range(num_layers)
        ])

    def forward(
            self,
            hidden_states,
            encoder_hidden_states,
            decoder_padding_mask: torch.Tensor = None,
            encoder_padding_mask: torch.Tensor = None
        ):
        for layer in self.layers:
            hidden_states = layer(
                hidden_states,
                encoder_hidden_states=encoder_hidden_states,
                decoder_padding_mask=decoder_padding_mask,
                encoder_padding_mask=encoder_padding_mask
            )

        return hidden_states
    
# testing
decoder = TransformerDecoder(512, 8, 128, 6, 0.1)
decoder

TransformerDecoder(
  (layers): ModuleList(
    (0-5): 6 x DecoderBlock(
      (causal_self_attention): MultiheadAttention(
        (w_q): Linear(in_features=512, out_features=512, bias=True)
        (w_k): Linear(in_features=512, out_features=512, bias=True)
        (w_v): Linear(in_features=512, out_features=512, bias=True)
        (w_o): Linear(in_features=512, out_features=512, bias=True)
        (attn_dropout): Dropout(p=0.1, inplace=False)
      )
      (cross_attention): MultiheadAttention(
        (w_q): Linear(in_features=512, out_features=512, bias=True)
        (w_k): Linear(in_features=512, out_features=512, bias=True)
        (w_v): Linear(in_features=512, out_features=512, bias=True)
        (w_o): Linear(in_features=512, out_features=512, bias=True)
        (attn_dropout): Dropout(p=0.1, inplace=False)
      )
      (ffn): Sequential(
        (0): Linear(in_features=512, out_features=2048, bias=True)
        (1): ReLU()
        (2): Linear(in_features=2048, out_features=

## 5. Complete Transformer

The model uses one token embedding matrix for the source and target. The language-model head is tied to that same matrix, matching the weight-sharing scheme described in the original Transformer.

The final hidden state at every decoder position is projected to one logit per vocabulary token. Cross-entropy compares those logits with the shifted labels, ignores padded label positions, and uses the original paper's label-smoothing value of $\epsilon_{ls}=0.1$.

In [ ]:
class Transformer(nn.Module):
    def __init__(
            self,
            vocab_size,
            embedding_dim,
            num_heads,
            context_length,
            num_layers,
            pad_token_id,
            dropout_rate=0.1,
            label_smoothing=0.1,
    ):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.pad_token_id = pad_token_id
        self.label_smoothing = label_smoothing
        self.sqrt_d_model = embedding_dim ** 0.5

        # input embeddings
        self.token_embeddings = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_token_id
        )
        self.position_encodings = SinusoidalPositionEmbedding(
            context_length=context_length,
            embedding_dim=embedding_dim
        )
        self.embedding_dropout = nn.Dropout(dropout_rate)

        # stacked encoder
        self.encoder = TransformerEncoder(
            embedding_dim=embedding_dim,
            num_heads=num_heads,
            context_length=context_length,
            num_layers=num_layers,
            dropout_rate=dropout_rate
        )

        # stacked decoder
        self.decoder = TransformerDecoder(
            embedding_dim=embedding_dim,
            num_heads=num_heads,
            context_length=context_length,
            num_layers=num_layers,
            dropout_rate=dropout_rate
        )

        # output
        self.lm_head = nn.Linear(embedding_dim, vocab_size, bias=False)

        # initialize the weights
        self.apply(self._init_weights)
        self.lm_head.weight = self.token_embeddings.weight
        with torch.no_grad():
            self.token_embeddings.weight[pad_token_id].zero_()

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=module.embedding_dim**-0.5)

    def _embed(self, token_ids: torch.Tensor):
        seq_len = token_ids.size(1)
        tok_embeds = self.token_embeddings(token_ids)
        positions = torch.arange(seq_len, device=token_ids.device)
        pos_embeds = self.position_encodings(positions)

        return self.embedding_dropout(
            tok_embeds * self.sqrt_d_model + pos_embeds
        )

    def forward(
            self,
            source_ids: torch.Tensor,
            decoder_input_ids: torch.Tensor,
            source_padding_mask: torch.Tensor,
            decoder_padding_mask: torch.Tensor,
            labels: torch.Tensor = None
    ):
        if source_padding_mask is None:
            source_padding_mask = source_ids.eq(self.pad_token_id)
        if decoder_padding_mask is None:
            decoder_padding_mask = decoder_input_ids.eq(self.pad_token_id)

        encoder_input = self._embed(source_ids)
        decoder_input = self._embed(decoder_input_ids)

        encoder_hidden_states = self.encoder(encoder_input, padding_mask=source_padding_mask)
        decoder_hidden_states = self.decoder(
            decoder_input,
            encoder_hidden_states=encoder_hidden_states,
            decoder_padding_mask=decoder_padding_mask,
            encoder_padding_mask=source_padding_mask
        )

        logits = self.lm_head(decoder_hidden_states)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                input=logits,
                target=labels,
                ignore_index=self.pad_token_id,
                label_smoothing=label_smoothing
            )

        return {"logits": logits, "loss": loss}
    
# testing
transformer = Transformer(1000, 128, 8, 128, 6, 0, 0.1, 0.1)
transformer

Transformer(
  (token_embeddings): Embedding(1000, 128, padding_idx=0)
  (position_encodings): SinusoidalPositionEmbedding()
  (embedding_dropout): Dropout(p=0.1, inplace=False)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-5): 6 x EncoderBlock(
        (self_attention): MultiheadAttention(
          (w_q): Linear(in_features=128, out_features=128, bias=True)
          (w_k): Linear(in_features=128, out_features=128, bias=True)
          (w_v): Linear(in_features=128, out_features=128, bias=True)
          (w_o): Linear(in_features=128, out_features=128, bias=True)
          (attn_dropout): Dropout(p=0.1, inplace=False)
        )
        (ffn): Sequential(
          (0): Linear(in_features=128, out_features=512, bias=True)
          (1): ReLU()
          (2): Linear(in_features=512, out_features=128, bias=True)
        )
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (ffn_dropout): Dropout(p=0.1, inplace=False)
        (layer_norm1): LayerNorm((128,